# Eight QML experiments: GroupMemory versus equally informed Qiskit

This notebook reads the saved 16-condition local CPU benchmark. **Run All launches no simulations.** It validates the published summaries and reconstructs the figures from numeric records. The runnable experiments and exact resume command are in `docs/eight_experiments.md`.

Expected readout runtime: seconds. Inputs: `results/eight-experiments/`. Checkpoints for the original run: `results/eight-experiments/checkpoints/manifest.json`. To conduct a new run, use a new output directory; the original run deadline is immutable.

The primary comparison gives Qiskit the **same symmetry, equivalent jobs, native compilation and simulator**. Each method is measured in a fresh process, including startup, imports, computation and output writing. All examples are synthetic ideal simulations; none ran on a QPU.

## Load the measured evidence

Five data seeds and three timing repeats are planned for each of sixteen conditions. A seed enters a comparison only when all matching quantum trials passed. The saved vectors must agree within absolute tolerance $10^{-10}$. Related views are not independent test samples.

In [ ]:
from pathlib import Path
import json, csv
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
DATA = ROOT / 'results' / 'eight-experiments'
analysis = json.loads((DATA / 'analysis.json').read_text())
config = json.loads((DATA / 'config.json').read_text())
summary = analysis['summary']
comparisons = analysis['comparisons']
assert analysis['successful'] == analysis['planned']
assert analysis['validation_passed'] == analysis['validation_planned']
assert analysis['maximum_absolute_error'] <= 1e-10
print(f"{analysis['successful']} successful attempts; {analysis['validation_passed']} matched blocks")
print(f"Maximum output disagreement: {analysis['maximum_absolute_error']:.3g}")
environment = json.loads((DATA / 'environment.json').read_text())
print('Environment:', environment['platform'], environment['machine'], '; one thread per sequential worker; ideal CPU simulation')


## What actually won?

A descriptive win requires at least five complete seeds, every seed median faster, and median paired saving of at least 5%. This is a fixed descriptive rule, not a statistical confidence or universal crossover claim. A startup-dominated win over a different classical implementation does not establish a learning advantage.

In [ ]:
primary = [r for r in comparisons if r['baseline'] == 'qiskit_symmetry']
winners = [r['condition'] for r in primary if r['descriptive_win']]
display(Markdown('**Eligible total-process wins over equally informed Qiskit:** ' + (', '.join(winners) if winners else 'none.')))
rows = ['|Condition|Group seconds|Symmetry-Qiskit seconds|Classical seconds|Group/classical task score|', '|---|---:|---:|---:|---|']
for condition in config['conditions']:
    lookup = {r['method']:r for r in summary if r['condition'] == condition['id']}
    g,q,c = (lookup[m] for m in ('group_memory','qiskit_symmetry','classical'))
    rows.append(f"|{condition['id']}|{g['process_median_s']:.3f}|{q['process_median_s']:.3f}|{c['process_median_s']:.3f}|{g['metric_name']}: {g['metric_mean']:.4g} / {c['metric_mean']:.4g}|")
display(Markdown('\n'.join(rows)))


## Paired speed ratios across seeds

Each point is the median of five seed-level ratios. Each line spans the observed minimum and maximum seed ratio; **it is not a confidence interval**. Each seed ratio uses three matched fresh-process repetitions. Values above one favor GroupMemory. The equally informed baseline is the primary test; ordinary exact caching is an ablation.

In [ ]:
ids = [c['id'] for c in config['conditions']]
fig, axes = plt.subplots(1, 2, figsize=(12, 8), constrained_layout=True)
for ax, baseline, color in zip(axes, ['qiskit_symmetry','qiskit_exact'], ['#126e82','#b56c21']):
    rows = {r['condition']:r for r in comparisons if r['baseline'] == baseline}
    for i, cid in enumerate(ids):
        ratios = np.asarray(json.loads(rows[cid]['seed_speedups']))
        ax.plot([ratios.min(), ratios.max()], [i,i], color=color, linewidth=2)
        ax.scatter(np.median(ratios), i, color=color, marker='o', s=35)
    ax.axvline(1, color='black', linestyle='--')
    ax.set_yticks(range(len(ids)),ids); ax.invert_yaxis()
    ax.set_xscale('log'); ax.set_xlabel('Baseline / GroupMemory total process time')
    ax.set_title(baseline.replace('_',' '))
fig.savefig(DATA / 'paired_seed_ranges.png', dpi=150)
plt.show()


## Learning quality

Accuracy is higher-is-better, with balanced chance at 50%. Mean squared error is lower-is-better. Scores average independent data seeds, not timing repetitions. These small synthetic test sets do not establish real-world generalization.

1A and 2A update quantum parameters; 2B uses Hessian updates. The other experiments train classical heads or SVMs over fixed features. In 4B, the analytic classical method computes the **same feature model** as the quantum cone simulation.

In [ ]:
methods = ['group_memory','qiskit_symmetry','qiskit_exact','classical']
labels = ['GroupMemory','Symmetry Qiskit','Exact-cache Qiskit','Classical']
colors = ['#126e82','#b56c21','#777777','#7b4587']
fig, axes = plt.subplots(1, 2, figsize=(13, 7), constrained_layout=True)
for ax, want_accuracy in zip(axes,[True,False]):
    chosen = [cid for cid in ids if ('accuracy' in next(r['metric_name'] for r in summary if r['condition']==cid)) == want_accuracy]
    for j,(method,label,color) in enumerate(zip(methods,labels,colors)):
        lookup = {r['condition']:r for r in summary if r['method']==method}
        vals = [lookup[c]['metric_mean'] * (100 if want_accuracy else 1) for c in chosen]
        ax.scatter(vals,np.arange(len(chosen))+(j-1.5)*.18,label=label,color=color,marker=['o','s','^','D'][j],s=28)
    ax.set_yticks(range(len(chosen)),chosen);ax.invert_yaxis()
    if want_accuracy:
        ax.set_xlim(0,105);ax.axvline(50,color='black',linestyle='--');ax.set_xlabel('Held-out accuracy (%)')
    else:
        ax.set_xscale('log');ax.set_xlabel('Held-out mean squared error')
    ax.legend(fontsize=8)
fig.savefig(DATA / 'learning_quality.png',dpi=150)
plt.show()


## Simulated CNOT work

These counts cover gates in preparations/evaluations actually simulated. They are not hardware jobs, and avoiding repeated evaluations does not shorten each native circuit. The equally informed Qiskit wrapper must receive identical work savings. Kernel overlaps and reverse gradients are classical operations and are excluded from these gate counts.

In [ ]:
with (DATA / 'seed_blocks.csv').open() as f:
    seed_blocks = list(csv.DictReader(f))
rows=['|Condition|Group simulated CX|Symmetry Qiskit CX|Exact-cache Qiskit CX|Median CX saving vs exact cache|','|---|---:|---:|---:|---:|']
for cid in ids:
    lookup={r['method']:r for r in summary if r['condition']==cid}
    g,q,e=(lookup[m]['total_cx_median'] for m in methods[:3])
    seeds={int(r['seed']) for r in seed_blocks if r['condition']==cid}
    b={(int(r['seed']),r['method']):r for r in seed_blocks if r['condition']==cid}
    savings=[1-float(b[(s,'group_memory')]['total_cx'])/float(b[(s,'qiskit_exact')]['total_cx']) for s in seeds if float(b[(s,'qiskit_exact')]['total_cx'])>0]
    assert g==q
    rows.append(f"|{cid}|{g:g}|{q:g}|{e:g}|{100*np.median(savings):.1f}%|")
display(Markdown('\n'.join(rows)))


## Accounting and limitations

The primary total clock includes interpreter startup, imports, source hashing, data generation, compilation, canonicalization, simulation, training, predictions, result writing and shutdown. The subsidiary application clock includes lazy imports and is not a clean steady-state compute benchmark. Package installation is excluded. Independent tests, the checkpoint pilot, parent orchestration and readout creation are recorded separately.

The 256-site experiment uses a connected shallow ring with exact three-qubit backward cones; it does not construct a 256-qubit statevector. Its analytic classical control is an exact model-equivalent comparison. Reused finite-shot estimates on hardware would introduce correlated errors, and noise can break ideal symmetries. Those effects are not tested here.

In [ ]:
print(f"Sum of all measured fresh-process walls: {analysis['process_wall_sum_s']:.3f} seconds")
print(f"Campaign wall including its parent/journal: {analysis['status']['elapsed_since_first_start_s']:.3f} seconds")
receipt = DATA / 'suite_wall_accounting.json'
if receipt.exists():
    print(json.dumps(json.loads(receipt.read_text()), indent=2))
